In [1]:
# NEEDED FOR RESAMPLING USING TORCHAUDIO
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# import torch

# torch.set_num_threads(1)
# torch.set_num_interop_threads(1)

In [21]:
import sys
from tqdm import tqdm

BASE_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr"
VOCAB_FILE = f"{BASE_PATH}/src/model/xeusphoneme/resources/ipa_vocab.json"

sys.path.append(BASE_PATH)

# ipapack
from src.data.kaldi_pretraining_dataset import build_kaldi_datamodule

datamodule = build_kaldi_datamodule(
    train_split="dev_1k",  # "train_accentmix_multi",
    dev_splits=[
        # "dev_1k",
        "dev_gmuaccent",
        # "dev_buckeye",
        # "dev_epadb",
        # "dev_speechoceanotth",
        # "dev_l2arctic",
    ],
    predict_split="predict",
    dataset_config_path=f"{BASE_PATH}/configs/data/ipapack_index.yaml",
    batch_size=1,
    num_workers=1,
    vocab_file=VOCAB_FILE,
    # limit_samples=2,
)

# from src.data.kaldi_dataset import build_kaldi_datamodule

# DATASET = "buckeye"
# datamodule = build_kaldi_datamodule(
#     DATASET,
#     data_dir="/work/hdd/bbjs/shared/powsm/s2t1/dump/raw",
#     dataset_config_path=f"{BASE_PATH}/configs/data/powsm_evalset_index.yaml",
#     portable_wavscp=False,
#     sampling_rate=16000,
#     batch_size=1,
#     num_workers=1,
#     vocab_file=VOCAB_FILE,
# )

datamodule.setup()
dataloader = datamodule.val_dataloader()
dataloader = iter(dataloader)

print("Loaded dataset with length:", len(dataloader))

Loaded 4655 samples from wav.scp: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/dev_1k_fixed/wav.scp with count 0


Reading text: 18620it [00:00, 595630.96it/s]
Reading language: 0it [00:00, ?it/s]

Reading language: 18620it [00:00, 603865.62it/s]


Loaded 318 samples from wav.scp: /work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/data/devsets/gmuaccent.scp with count 0


Reading text: 1242it [00:00, 11951.09it/s]
Reading language: 1242it [00:00, 43688.47it/s]


Loaded 4655 samples from wav.scp: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/dev_1k_fixed/wav.scp with count 0


Reading text: 18620it [00:00, 630773.35it/s]
Reading language: 18620it [00:00, 616084.41it/s]


Loaded dataset with length: 318


In [22]:
import torch
from src.model.wav2vec2.builders import build_wav2vec2pr_inference
import json

# CKPT_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/ipaaccent_ctc/mms_multiaccent.bs256.lr1em5/checkpoints/checkpoint-500.ckpt"
CKPT_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/ipaaccent_ctc/mms_multiaccent.bs256.lr5em5/checkpoints/checkpoint-48500.ckpt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device = ", device)

inference = build_wav2vec2pr_inference(
    hf_repo="facebook/mms-300m",
    vocab_file=VOCAB_FILE,
    checkpoint=CKPT_PATH,
)

with open(VOCAB_FILE, "r") as f:
    vocab = json.load(f)
id2token = {k: v for v, k in vocab.items()}

Using device =  cuda


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-300m and are newly initialized: ['lm_head.bias', 'lm_head.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Sampling rate: 16000
Loaded checkpoint: /work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/ipaaccent_ctc/mms_multiaccent.bs256.lr5em5/checkpoints/checkpoint-48500.ckpt with load info: <All keys matched successfully>


In [23]:
from src.metrics.phone_recognition import PhoneRecognitionEvaluator

evaluator = PhoneRecognitionEvaluator()


def get_phone_str(token_ids):
    return "/".join([id2token[t] for t in token_ids if t in id2token])


N_SAMPLES = 150
n_data = len(dataloader)
print("Total data samples:", n_data, "sampling ", N_SAMPLES)
results = []
with torch.no_grad():
    for bidx, batch in tqdm(enumerate(dataloader), desc="Making predictions"):
        # if bidx % (n_data // min(N_SAMPLES, n_data)) != 0:
        #     continue
        # assert batch size is 1
        # print(batch)
        batch, _, _ = batch
        batch = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }

        prediction = inference(**batch)
        for i, key in enumerate(batch["keys"]):
            pred = prediction[i]["processed_transcript"]
            gt_str = batch["text"][i]
            if isinstance(gt_str, torch.Tensor):
                gt_str = gt_str.cpu().numpy()
                gt_phones = get_phone_str(gt_str)
            else:
                gt_phones = gt_str
            prmetrics, _ = evaluator.evaluate(
                {i: {"prediction": pred, "transcription": gt_phones.replace("/", "")}},
                compute_inventory=False,
            )
            asr_text = batch["asr_text"][i] if "asr_text" in batch else ""
            results.append(
                {
                    "key": key,
                    "speech": batch["speech"][i].cpu().numpy(),
                    "speech_length": batch["speech_length"][i].cpu().item(),
                    "wavpath": batch["wavpath"][i],
                    "phone_str": gt_phones,
                    "language": batch["lang_sym"][i],
                    "asr_text": asr_text,
                    "prediction": prediction[i]["predicted_transcript"],
                    "pr_metrics": prmetrics,
                }
            )
        if bidx > 1:
            break
print(f"Collected {len(results)} samples for error analysis.")

Total data samples: 318 sampling  150


Making predictions: 2it [00:40, 20.21s/it]

Collected 3 samples for error analysis.


In [24]:
results

[{'key': 'rundi1',
  'speech': array([ 3.2170680e-03,  5.0967387e-03,  4.5999140e-03, ...,
         -3.0531770e-05, -3.0531777e-05, -3.0531908e-05],
        shape=(320000,), dtype=float32),
  'speech_length': 320000,
  'wavpath': '/work/hdd/bbjs/shared/powsm/s2t1/dump/raw/test_gmuaccent/recording/rundi1.wav',
  'phone_str': 'p/l/iː/z/k/ɑ/l/s/t/ɛ/l/a/æ̞/s/k/h/ɜ/t/u/b/ɹ/ɪ̃/ŋ/ð/iː/z/θ/i/n/k/s/w/ɪ/θ/h/ə˞/f/ɹ/ʌ̃/m/ð/ɛ/s/t/ɔː/ɹ/s/ɪ/k/s/s/p/<unk>/n/z/ə/v/f/ɹ/ɛ/ʃ/s/n/o/ʊ/p/iː/z/f/a/v/θ/i/k/s/l/æ/b/s/ə/v/b/l/uː/t/ʃ/iː/z/æ̃/n/d/m/e/i/b/i/ɛ/s/n/æ/k/f/ɔ/h/ɜ/b/ɹ/ʌ/ð/ə˞/b/o/b/w/i/ɑ/l/s/o/n/iː/d/ə/s/m/ɑ/l/p/l/æ/s/t/ɪ/k/s/n/eː/ɪ/k/æ̃/n/d/e/b/i/k/tʰ/ɔ/ɪ/f/ɹ/ɑ/ɡ/f/ɔ/ɹ/ð/ə/k/i/d/s/ʃ/i/k/<unk>/n/s/k/uː/p/ð/i/s/θ/ɪ/n/k/s/ɪ̃/n/t/u/θ/r/iː/ɹ/ɛ/d/b/æ/k/s/æ̃/n/d/w/i/w/ɪ/ɡ/o/ʊ/m/iː/t/h/ɜː/ɹ/w/ɛ̃/n/z/n/e/ɪ/æ̞/t/ð/ə/t/ɹ/e/ĩ/n/s/t/e/ʃ/ə̃/n',
  'language': 'eng',
  'asr_text': None,
  'prediction': 'pʰ/l/i/z/kʰ/ɑ/l/s/t/ɪ/l/ə/æ/s/k/h/ə˞/tʰ/u/p/ɹ/ɪ̃/ŋ/ð/i/z/θ/ɪ̃/ŋ/z/w/ɪ/ð/h/ə˞/f/ɹ/ʌ/m/ð/ə/s/t/ɔ/ɹ/s/ɪ/k/s/p/ũ/n/z/ʌ/v

In [25]:
def play_audio(sp):
    import IPython.display as ipd

    return ipd.Audio(sp, rate=16000)

In [26]:
for res in results:
    print("WAVPATH:", res["wavpath"])
    print("Language:", res["language"])
    print("ASR Text:", res["asr_text"])
    print("Ground Truth Phones:", res["phone_str"])
    print("Predicted Phones:", res["prediction"])
    print("PR Metrics:", res["pr_metrics"])
    display(play_audio(res["speech"]))
    print("-" * 40)

WAVPATH: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/test_gmuaccent/recording/rundi1.wav
Language: eng
ASR Text: None
Ground Truth Phones: p/l/iː/z/k/ɑ/l/s/t/ɛ/l/a/æ̞/s/k/h/ɜ/t/u/b/ɹ/ɪ̃/ŋ/ð/iː/z/θ/i/n/k/s/w/ɪ/θ/h/ə˞/f/ɹ/ʌ̃/m/ð/ɛ/s/t/ɔː/ɹ/s/ɪ/k/s/s/p/<unk>/n/z/ə/v/f/ɹ/ɛ/ʃ/s/n/o/ʊ/p/iː/z/f/a/v/θ/i/k/s/l/æ/b/s/ə/v/b/l/uː/t/ʃ/iː/z/æ̃/n/d/m/e/i/b/i/ɛ/s/n/æ/k/f/ɔ/h/ɜ/b/ɹ/ʌ/ð/ə˞/b/o/b/w/i/ɑ/l/s/o/n/iː/d/ə/s/m/ɑ/l/p/l/æ/s/t/ɪ/k/s/n/eː/ɪ/k/æ̃/n/d/e/b/i/k/tʰ/ɔ/ɪ/f/ɹ/ɑ/ɡ/f/ɔ/ɹ/ð/ə/k/i/d/s/ʃ/i/k/<unk>/n/s/k/uː/p/ð/i/s/θ/ɪ/n/k/s/ɪ̃/n/t/u/θ/r/iː/ɹ/ɛ/d/b/æ/k/s/æ̃/n/d/w/i/w/ɪ/ɡ/o/ʊ/m/iː/t/h/ɜː/ɹ/w/ɛ̃/n/z/n/e/ɪ/æ̞/t/ð/ə/t/ɹ/e/ĩ/n/s/t/e/ʃ/ə̃/n
Predicted Phones: pʰ/l/i/z/kʰ/ɑ/l/s/t/ɪ/l/ə/æ/s/k/h/ə˞/tʰ/u/p/ɹ/ɪ̃/ŋ/ð/i/z/θ/ɪ̃/ŋ/z/w/ɪ/ð/h/ə˞/f/ɹ/ʌ/m/ð/ə/s/t/ɔ/ɹ/s/ɪ/k/s/p/ũ/n/z/ʌ/v/f/ɹ/ɛ/ʃ/s/n/o/ʊ/pʰ/i/z/f/a/ɪ/v/θ/ɪ/k/s/l/æ/b/z/ʌ/v/p/l/u/t/ʃ/i/z/ə/n/d/m/e/ɪ/p/i/ə/s/n/ʌ/k/f/ɔ/ɹ/h/ə˞/p/ɹ/ʌ/ð/ə˞/p/ɑ/b/w/i/ɔ/l/s/o/ʊ/n/i/ð/ə/s/m/ɔ/l/pʰ/l/æ/s/t/ɪ/k/s/n/e/ɪ/k/ə/n/d/ə/p/ɪ/ɡ/t/ə˞/f/ɹ/ɑ
PR Metrics: PhoneRecognitionSumm

----------------------------------------
WAVPATH: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/test_gmuaccent/recording/english152.wav
Language: eng
ASR Text: None
Ground Truth Phones: pʰ/l/iː/s/kʰ/o/lˠ/s/t/ɛ/l/ə/a/s/k/ɜ/ɾ/ə/b/ɹ/ɪ̃/ŋ/n̪/i/z/θ/ɪ̃/ŋ/z/w/ɪ/ð/ɜ/f/ɹ/ə̃/m/n̪/ə/s/t/ɔː/s/<unk>/k/s/s/p/<unk>/n/z/ə/v/f/ɹ/ɛ/ʃ/s/n/ə/ʊ/pʰ/iː/z/f/a/ɪ/v/θ/<unk>/k/s/l/æ̝/b/z/ə/v/b/l/<unk>/tʰ/ʃ/iː/z/ɛ̃/m/m/e̞/ɪ/b/i/ɪ/s/n/æ̝/k/f/ɜ/h/<unk>/b/ɹ/ʌ/ð/ə/b/ɑ/b/w/i/ɒ/lˠ/s/ə/n/iː/d/ə/s/m/ɒː/lˠ/pʰ/l̠/æ̝/s/t/ɪ/k/s/n/e/ɪ/k/ɛ̃/n/ə/b/ɪ/ɡ/tʰ/ɔ/ɪ/f/ɹ/ɔː/ɡ/f/ə/ð/ə/kʰ/ɪ/<unk>/d/z/ʃ/i/kʰ/ə̃/n/s/k/<unk>/p/ð/i/ð/θ/ɪ̃/ŋ/z/ɪ̃/n/t/ə/θ/ɹ/iː/ɹ/ɛ/d/b/æ̝/ɡ/z/ə̃/n/w/ɪ/lˠ/ɡ/ə/<unk>/ɪ̃/n/m/i/t/h/ɜ/w/ɛ̃/n/z/e/ɪ/æ/t/ð/ə/t/ɹ/e̞/ɪ̃/n/s/t/e/ɪ/ʃ/ə̃/n
Predicted Phones: pʰ/l/i/z/kʰ/u/l/d/s/t/e/ɪ/l/ə˞/æ/s/k/h/ə˞/tʰ/u/p/ɹ/ɪ̃/ŋ/ð/i/z/θ/ɪ̃/ŋ/z/w/ɛ/ð/ə˞/f/ɹ/ʌ/m/ð/ə/s/t/ɔ/ɹ/s/ɪ/k/s/s/p/ũ/n/d/z/ʌ/v/f/ɹ/ɛ/ʃ/s/n/o/ʊ/f/i/z/f/a/ɪ/v/θ/ɪ/k/s/l/æ/b/z/ʌ/v/p/l/u/t/ʃ/i/z/ə/n/d/m/e/ɪ/p/i/ə/s/n/æ/k/f/ɔ/ɹ/h/ə˞/p/ɹ/ʌ/ð/ə˞/p/ɑ/b/w/i/ɔ/l/s/o/ʊ/n/i/d/ə/s/m/ɔ/l/pʰ/l/

----------------------------------------
WAVPATH: /work/hdd/bbjs/shared/powsm/s2t1/dump/raw/test_gmuaccent/recording/turkish2.wav
Language: eng
ASR Text: None
Ground Truth Phones: p/<unk>/l/iː/z/k/ɔ/s/t/ɛ/l/a/a/s/k/h/a/s/k/h/t/<unk>/b/ə/r/ĩ/ŋ/d̪/ɪ/s/t̪/ĩ/ŋ/s/w/ɪ/d/ɪ/d/w/ɪ/t̪/h/f/ɹ/o/m/d̪/ə/s/t/ɔ/ɹ/s/ɪ/k/s/s/p/ʊ̃/s/s/ɪ/k/s/s/p/ʊ/n/s/o/f/f/ɹ/ɛ̝/ʃ/s/n/o/p/iː/z/f/a/ɪ/f/t/i/k/s/ə/l/a/p/s/ɔ/f/d̪/ə/b/ə/l/u/i/z/æ/n/m/e/b/i/e/s/n/e/k/f/o/ɹ/d̪/ə/b/ɹ/<unk>/b/ɹ/ʌ/d̪/w/i/o/l/s/o/v/i/o/l/s/o/n/i/d/ə/s/m/ɑ/l/p/l/a/s/t/ɪ/k/s/n/e/k/<unk>/n/ə/<unk>/n/e/b/i/k/tʰ/ɔ/ɪ/f/ɹ/ɑ/ɡ/f/o/ɹ/d̪/ə/k/i/d/s/ʃ/i/k/æ̃/s/k/ʊ/p/d̪/i/z/t̪/i/ŋ/s/ɪ/n/t/u/t̪/ɹ/i/b/æ/t̪/ɹ/iː/b/ɛ/ɡ/z/æ/n/d/w/ɪ/l/ɡ/o/m/i/tʷ/w/ɛ̃/n/ə/z/d/e/æ/t/t̪/ə/t/ɹ/e/n/s/tʷ/e/ʃ/ɪ/n
Predicted Phones: pʰ/l/ɪ/s/k/o/ʊ/s/t/ɪ/l/ə/æ/s/k/h/ə˞/æ/s/k/h/ə˞/tʰ/u/p/ɪ/ɹ/ɪ̃/ŋ/t/ɪ/s/t/ɪ̃/ŋ/z/w/ɪ/ð/w/ɪ/ð/h/ə˞/f/ʌ/m/ð/ə/s/t/ɔ/ɹ/s/ɪ/k/s/p/ə/n/s/s/ɪ/k/s/s/p/ũ/n/z/ɔ/f/ɹ/ɪ/s/ə/n/ə/pʰ/i/z/f/a/ɪ/v/tʰ/ɪ/s/ə/l/ə/p/s/ʌ/v/ð/ə/p/l/u/t/ʃ/i/z/ə/n/d/m/e/ɪ/p/i/ə/s/ɪ̃/n/ɪ/k/f/ɔ/ɹ/ð/ə/p/ɹ/p/

----------------------------------------
